In [1]:
#                    #
##                  ##
######################
### Build datacard ###
######################
######################

import numpy as np
import pandas as pd
import json
import os
import sys
import functools
import json

def getZhbbWeight(df_, year):
    tot_weight = (df_['norm_weight'] * np.sign(df_['genWeight']) * df_['topptWeight'] * 
                  df_['ele_reco_sf'] * df_['ele_id_sf'] * df_['mu_id_sf'] * df_['mu_iso_sf'] *
                  df_['xbb_sf'] * df_['btag_sf'] * df_['mu_reco_sf'] * df_['mu_trig_sf'] * df_['ele_trig_sf'] *
                  df_['puWeight'] * df_['l1prefire_sf']) # and other weights
    return tot_weight
pt_bins = [0,200,300,450, np.inf]
#pt_bins = [200,300,450]
mass_bins = [50,80,105,145,200]
#mass_bins = [50,80,105,145]
isblind = True

In [2]:
def cuts(df_):
    base = (
    (df_['n_ak4jets']   >= 5)       &
    (df_['n_b_outZH'] >= 2) &
    (df_['ZH_bbvLscore'] >= 0.9172) &
    #(df_['ZH_pt']       >= 200)& # 200
    (df_['MET_pt']      >= 20)            &
    (df_['ZH_M']        >= 50)            &
    (df_['ZH_M']        <= 200)
    )
    return base

def stxs_cut(df_):
    base = (
        (df_['ZH_rap'] <= 2.5)
    )
    return base

In [9]:
def calculate_systematics(target_df, nominal_stxs_yield, year, sample_name):
    sys_results = {}
    systematics = [
        'isr_up', 'isr_down', 'fsr_up', 'fsr_down', 
        'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down'
    ]
    
    # Pre-filter the dataframe to save computation time
    mask = (target_df['process'] == sample_name) & stxs_cut(target_df)
    filtered_df = target_df[mask]
    
    for sys in systematics:
        # Only calculate if the column actually exists in the dataframe
        if sys in target_df.columns:
            if nominal_stxs_yield != 0:
                # Calculate the varied yield (denominator)
                denom = np.sum(getZhbbWeight(filtered_df, year) * filtered_df[sys])
                # Safely divide, defaulting to 1.0 if denom is 0
                sys_results[sys] = (nominal_stxs_yield / denom) if denom != 0 else 1.0
            else:
                sys_results[sys] = 1.0
                
    return sys_results

In [10]:
samples = ['ttH', 'ttZ', 'TTBar', 'tt_B', 'ttX', 'VJets', 'SingleTop']
get_pickle= (lambda s: pd.read_pickle(f'pickled/Nominal2018/{s}.pkl'))
df = pd.concat([get_pickle(s) for s in samples], axis='rows', ignore_index=True)
#df = df[cuts(df)]
outer_dict = {}
inner_dict = {}

pt_bins = [0, 200, 300, 450, np.inf]

for y in ['2018']:
    for sample in samples:
        _df = df[cuts(df)].copy() 
        _df['pt_bin'] = pd.cut(
            _df['genZHpt'], 
            bins=pt_bins, 
            labels=[f'Zhpt{i}' for i in range(len(pt_bins) - 1)]
        )
        
        # Calculate inclusive yields
        sumw = np.sum(getZhbbWeight(_df[_df['process'] == sample], y))
        sumw_stxs = np.sum(getZhbbWeight(_df[(_df['process'] == sample) & stxs_cut(_df)], y))
        
        process_dict = {
            "yield" : sumw,
            "stxs_yield": sumw_stxs,
        }
        
        # --- NEW: Dynamically add systematics to inclusive dict ---
        sys_dict = calculate_systematics(_df, sumw_stxs, y, sample)
        process_dict.update(sys_dict) 
        
        if sample == 'tt_B':
            inner_dict['ttbb'] = process_dict
        else:
            inner_dict[sample] = process_dict
            
        # ---------------------------------------------------------
        # Differential Yields (ttH and ttZ only)
        # ---------------------------------------------------------
        if sample in ['ttH', 'ttZ']:
            for i in range(len(pt_bins) - 1):
                bin_df = _df[_df['pt_bin'] == f'Zhpt{i}']
                
                bin_sumw = np.sum(getZhbbWeight(bin_df[bin_df['process'] == sample], y))
                bin_sumw_stxs = np.sum(getZhbbWeight(bin_df[(bin_df['process'] == sample) & stxs_cut(bin_df)], y))
                
                bin_process_dict = {
                    "yield" : bin_sumw,
                    "stxs_yield": bin_sumw_stxs,
                }
                
                # --- NEW: Dynamically add systematics to differential dict ---
                bin_sys_dict = calculate_systematics(bin_df, bin_sumw_stxs, y, sample)
                bin_process_dict.update(bin_sys_dict)
                
                inner_dict[f'{sample}{i}'] = bin_process_dict

    outer_dict[y] = inner_dict

In [11]:
outer_dict

{'2018': {'ttH': {'yield': 83.79042156544728,
   'stxs_yield': 83.79042156544728,
   'isr_up': 1.0015946681190728,
   'isr_down': 0.9979527756674051,
   'fsr_up': 0.9810338445188476,
   'fsr_down': 1.0266534100141396,
   'mu_r_up': 1.0651424112671983,
   'mu_r_down': 0.9572979247123722,
   'mu_f_up': 1.0268024664444682,
   'mu_f_down': 0.9683926946107092},
  'ttH0': {'yield': 39.521341854215564,
   'stxs_yield': 39.521341854215564,
   'isr_up': 1.0023348542158772,
   'isr_down': 0.9969844886628183,
   'fsr_up': 0.9684882113345733,
   'fsr_down': 1.0405937729016475,
   'mu_r_up': 1.067616774492668,
   'mu_r_down': 0.9552766879157426,
   'mu_f_up': 1.024108903171243,
   'mu_f_down': 0.9706171286235883},
  'ttH1': {'yield': 18.221496199096343,
   'stxs_yield': 18.221496199096343,
   'isr_up': 1.0012050959305283,
   'isr_down': 0.9985867055402734,
   'fsr_up': 0.9907728505234267,
   'fsr_down': 1.025397206223308,
   'mu_r_up': 1.0654339651257452,
   'mu_r_down': 0.9567020922209716,
   'mu_

In [12]:
out_json_file = './process_norms/process_norms_2018.json'
with open(out_json_file, 'w') as jsf:
    json.dump(outer_dict, jsf, indent=4)

In [ ]:
df = df[cuts(df)]

In [ ]:
sumw = np.sum(getZhbbWeight(df, '2017'))

In [ ]:
process_dict = {
    "yield" : sumw
}

In [10]:
import pandas as pd
import glob
import numpy as np

# Load all generated pickle files
pkl_files = glob.glob("pickled/SpanetTests2018/*.pkl")

print(f"{'Process / File':<30} | {'Raw Events':<12} | {'Sum of norm_weight (Yield)':<25} | {'Missing/NaN Weights'}")
print("-" * 95)

total_bkg = 0
total_sig = 0

for f in pkl_files:
    df = pd.read_pickle(f)
    process = df['process'].iloc[0] if not df.empty else "Unknown"
    
    raw_events = len(df)
    
    # Check for NaNs or Infs in norm_weight
    nan_count = df['norm_weight'].isna().sum()
    inf_count = np.isinf(df['norm_weight']).sum()
    bad_weights = nan_count + inf_count
    
    # Calculate the base yield (just the luminosity and xsec scaling)
    # If you apply the sign of genWeight in your final datacard, include it here:
    if 'genWeight' in df.columns:
        yield_val = np.sum(df['norm_weight'] * np.sign(df['genWeight']))
    else:
        yield_val = np.sum(df['norm_weight'])
        
    if process in ['ttH', 'ttZ']:
        total_sig += yield_val
    elif process != 'data_obs':
        total_bkg += yield_val
        
    print(f"{process:<30} | {raw_events:<12,}| {yield_val:<25.2f} | {bad_weights}")

print("-" * 95)
print(f"Total Expected Signal:     {total_sig:,.2f}")
print(f"Total Expected Background: {total_bkg:,.2f}")

Process / File                 | Raw Events   | Sum of norm_weight (Yield) | Missing/NaN Weights
-----------------------------------------------------------------------------------------------
ttZ                            | 1,346,596   | 756.58                    | 0
TTBar                          | 3,165,467   | 140566.35                 | 0
tt_B                           | 190,317     | 8248.17                   | 0
data_obs                       | 2,183,354   | 1228840.00                | 0
ttH                            | 446,643     | 778.12                    | 0
-----------------------------------------------------------------------------------------------
Total Expected Signal:     1,534.70
Total Expected Background: 148,814.52
